# Edge features of high photos and of the map at the true place, and what scale carries the place (iteration 30)

**Question.** Above ~800 m single spots are about one pixel. Do larger structures take over (so a coarser edge scale should work better at height), or is the place information still in the finest edges? Look at the data first.

**Part 1 (figure).** Four development photos at about 600, 1000, 1400 and 1900 m (nearest to each target, no choice by success): raw photo, canvas (north-up, 224 px at the photo's map level), the canvas edge-direction cells, the map window at the TRUE place, its edge cells, and the same edge cells at a 4x coarser gradient scale. Colour = edge direction (hue), brightness = strength.

**Part 2 (scale study).** For every development photo of 500 m and higher: one query (heading 0, scale 1.0, hint jittered 5 km), the edge-direction ZNCC search over the 5.5 km disc with the gradient blur sigma = 1, 2, 3, 4, 6, 8 pixels (same code for canvas and map window). Per height band: share with the best peak within 2 sigma of the truth, share wrong (beyond 8 sigma), median z of the best peak. If larger structure takes over at height, a larger sigma wins there.

Development photos only (332). The 168 locked photos are not read.

Env: `EF_HEADLESS, EF_DATASET, EF_STORE, EF_LEVELS, EF_OUT, EF_SPLIT_DEV_ONLY`.

In [1]:
import os, sys, math, colorsys
import numpy as np, pandas as pd, torch, cv2
HEADLESS = os.environ.get("EF_HEADLESS", "../../../../Main_Pipeline_Headless"); sys.path.insert(0, os.path.abspath(HEADLESS))
import settings
settings.LEVELS = [int(v) for v in os.environ.get("EF_LEVELS", "3,4,5,6,7,8,9,10,11").split(",")]
from geoloc import photo, queries, scoring
from geoloc.cli import read_rows, move_hint
from geoloc.coarse_search import DEVICE, score_map_for_query, pick_peaks, window_transforms, zncc_everywhere, fast_length, cut_map_features
from geoloc.edge_features import normalise_cells
from geoloc.levels import level_for_height, level_gsd, map_piece
from geoloc.map_store import MapStore
CELL = settings.CELL; TILE = settings.TILE
DATASET = os.environ["EF_DATASET"]; STORE = os.environ["EF_STORE"]; OUT = os.environ.get("EF_OUT", "results/edge_features"); os.makedirs(OUT, exist_ok=True)
store = MapStore(STORE); rows = read_rows(DATASET)       # the dataset folder holds the development photos only
heights = np.array([float(r["alt"]) for r in rows]); print(len(rows), "photos, heights", heights.min(), heights.max())

332 photos, heights 100.599998 2016.599976


In [2]:
def edge_cells(image, sigma, cell_valid=None):
    # 8-direction edge strength summed over 4x4-pixel cells, gradient blur = sigma pixels (sigma 1 = the pipeline)
    blurred = cv2.GaussianBlur(image.astype(np.float32), (0, 0), sigma)
    gx = np.zeros_like(blurred); gy = np.zeros_like(blurred)
    gx[:, 1:-1] = blurred[:, 2:] - blurred[:, :-2]; gy[1:-1, :] = blurred[2:, :] - blurred[:-2, :]
    strength = np.hypot(gx, gy); angle = np.arctan2(gy, gx) % np.pi; bin_id = np.minimum((angle / np.pi * settings.N_BINS).astype(int), settings.N_BINS - 1)
    r, c = image.shape[0] // CELL, image.shape[1] // CELL; out = np.zeros((settings.N_BINS, r, c), np.float32)
    for b in range(settings.N_BINS):
        out[b] = (strength * (bin_id == b))[:r * CELL, :c * CELL].reshape(r, CELL, c, CELL).sum(axis=(1, 3))
    return out

def canvas_features(canvas, valid, sigma):
    safe = cv2.erode(valid.astype(np.uint8), np.ones((7, 7), np.uint8)); side = canvas.shape[0] // CELL
    cell_valid = safe.reshape(side, CELL, side, CELL).mean(axis=(1, 3)) >= 0.8
    return normalise_cells(edge_cells(canvas, sigma), cell_valid) * cell_valid, cell_valid

def window_pixels(level_image, top, left, cells_wide):
    size = cells_wide * CELL; r0, c0 = top * CELL, left * CELL; piece = np.zeros((size, size), np.float32); h, w = level_image.shape
    a0, a1 = max(r0, 0), min(r0 + size, h); b0, b1 = max(c0, 0), min(c0 + size, w)
    if a1 > a0 and b1 > b0: piece[a0 - r0:a1 - r0, b0 - c0:b1 - c0] = np.asarray(level_image[a0:a1, b0:b1], np.float32) / 255.0
    return piece

def map_features(piece, sigma):
    feats = edge_cells(piece, sigma); has = (piece[:feats.shape[1] * CELL, :feats.shape[2] * CELL].reshape(feats.shape[1], CELL, feats.shape[2], CELL).min(axis=(1, 3)) > 0)
    has = cv2.erode(has.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0
    feats = normalise_cells(feats, has) * has; return feats, has

def hsv_picture(feats):
    # hue = dominant edge direction, brightness = total strength (99th percentile = full)
    total = feats.sum(axis=0); hue = feats.argmax(axis=0) / settings.N_BINS; val = np.clip(total / (np.percentile(total, 99) + 1e-6), 0, 1)
    rgb = np.zeros(total.shape + (3,), np.float32)
    for i in range(total.shape[0]):
        for j in range(total.shape[1]): rgb[i, j] = colorsys.hsv_to_rgb(hue[i, j], 0.9, val[i, j])
    return rgb

In [3]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
dev = pd.DataFrame(dict(i=range(len(rows)), height=heights, name=[r["filename"] for r in rows]))
picked = [int(dev.iloc[(dev.height - h).abs().argsort().iloc[0]].i) for h in (600, 1000, 1400, 1900)]
fig, axes = plt.subplots(len(picked), 8, figsize=(24, 3.3 * len(picked)))
for row, index in enumerate(picked):
    meta = rows[index]; camera = photo.read_camera(meta); pose = photo.read_pose(meta); level = level_for_height(pose["height"], camera)
    truth = store.gps_to_metres(float(meta["lat_gtruth"]), float(meta["lon_gtruth"])); sigma = scoring.sigma_m(pose["height"], camera["fx"]); gsd = level_gsd(level)
    path = os.path.join(DATASET, "images", meta["filename"]); picture = photo.prepare_photo(path, pose, camera, level)
    canvas, valid = photo.draw_canvas(picture, pose, camera, level, 0.0, 1.0)
    top = int(round((truth[1] / gsd - TILE / 2) / CELL)); left = int(round((truth[0] / gsd - TILE / 2) / CELL))   # window position in CELLS
    piece = window_pixels(store.levels[level], top, left, TILE // CELL)
    raw = cv2.imread(path, cv2.IMREAD_UNCHANGED).astype(np.float32); lo, hi = np.percentile(raw, [1, 99]); raw = np.clip((raw - lo) / (hi - lo + 1e-6), 0, 1)
    cf1, cv_ = canvas_features(canvas, valid, 1.0); mf1, mh1 = map_features(piece, 1.0); cf4, _ = canvas_features(canvas, valid, 4.0); mf4, mh4 = map_features(piece, 4.0)
    mask = cv_ & mh1; a = cf1[:, mask].ravel(); b = mf1[:, mask].ravel(); cos1 = float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))
    mask4 = cv_ & mh4; a = cf4[:, mask4].ravel(); b = mf4[:, mask4].ravel(); cos4 = float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))
    panels = [(raw, "raw photo %s\n%.0f m, level %d (%.1f m/px)" % (meta["filename"][:6], pose["height"], level, gsd), "gray"), (canvas, "canvas (north-up, %d px)" % TILE, "gray"),
              (hsv_picture(cf1), "canvas edges, sigma 1 px\n(pipeline)", None), (piece, "MAP at the true place", "gray"), (hsv_picture(mf1), "map edges, sigma 1 px\ncosine to canvas %.2f" % cos1, None),
              (hsv_picture(cf4), "canvas edges, sigma 4 px", None), (hsv_picture(mf4), "map edges, sigma 4 px\ncosine to canvas %.2f" % cos4, None)]
    for col, (img, title, cmap) in enumerate(panels):
        axes[row, col].imshow(img, cmap=cmap, vmin=0 if cmap else None, vmax=1 if cmap else None); axes[row, col].set_title(title, fontsize=8); axes[row, col].axis("off")
    # score map of the edge search around the truth, to see how sharp the true peak is
    cache = {}; score_map, geometry = score_map_for_query(canvas, valid, level, store.gps_to_metres(float(meta["lat_sc"]), float(meta["lon_sc"])), store.features, cache)
    sm = score_map.cpu().numpy(); axes[row, 7].imshow(np.where(sm > -1, sm, np.nan), cmap="viridis"); axes[row, 7].set_title("edge ZNCC over the 5.5 km disc\n(photo hint jittered 0)", fontsize=8)
    xs = geometry["x_m"].cpu().numpy(); ys = geometry["y_m"].cpu().numpy(); cx = int(np.argmin(np.abs(xs - truth[0]))); cy = int(np.argmin(np.abs(ys - truth[1])))
    axes[row, 7].plot(cx, cy, "r+", ms=14, mew=2); axes[row, 7].axis("off")
fig.suptitle("Edge features of development photos at increasing height next to the map at the TRUE place (red + = truth in the score map)", y=1.0)
fig.tight_layout(); fig.savefig(os.path.join(OUT, "edge_features_by_height.png"), dpi=95); print("saved figure for photos", [rows[i]["filename"] for i in picked], [round(heights[i]) for i in picked])

saved figure for photos ['b9a4fab32e5c46e4be6a31fc6fdcaf45.png', 'e9646a11e3c04be6b6241c387c9faccb.png', 'b631c8037e0d41768db7bb42513c07b5.png', 'ebda9a66d3f6470d8113ea72ed28d4dc.png'] [603, 1002, 1402, 1886]


In [4]:
# ---- Part 2: scale study ----
SIGMAS = [1, 2, 3, 4, 6, 8]; records = []; query = queries.query_list()[0]
for index, meta in enumerate(rows):
    if float(meta["alt"]) < 500: continue
    meta = move_hint(meta, index, 5000.0, 0); camera = photo.read_camera(meta); pose = photo.read_pose(meta)
    prior = store.gps_to_metres(float(meta["lat_sc"]), float(meta["lon_sc"])); truth = store.gps_to_metres(float(meta["lat_gtruth"]), float(meta["lon_gtruth"]))
    sigma_m = scoring.sigma_m(pose["height"], camera["fx"]); level = level_for_height(pose["height"], camera); gsd = level_gsd(level)
    picture = photo.prepare_photo(os.path.join(DATASET, "images", meta["filename"]), pose, camera, level)
    canvas, valid = photo.draw_canvas(picture, pose, camera, level, query["heading"], query["scale_factor"]); side = TILE // CELL
    reach = settings.DISC_RADIUS / gsd + side * CELL / 2; cells_wide = int(math.ceil(2 * reach / CELL))
    top = int(round((prior[1] / gsd - reach) / CELL)); left = int(round((prior[0] / gsd - reach) / CELL))
    piece = window_pixels(store.levels[level], top, left, cells_wide); size = (fast_length(cells_wide), fast_length(cells_wide))
    row_ids = np.arange(cells_wide - side + 1); y_m = ((top + row_ids) * CELL + side * CELL / 2) * gsd; x_m = ((left + row_ids) * CELL + side * CELL / 2) * gsd
    inside = (x_m[None, :] - prior[0]) ** 2 + (y_m[:, None] - prior[1]) ** 2 <= settings.DISC_RADIUS ** 2
    near_truth = (x_m[None, :] - truth[0]) ** 2 + (y_m[:, None] - truth[1]) ** 2 <= (2 * sigma_m) ** 2
    for s in SIGMAS:
        mf, mh = map_features(piece, float(s)); kernel, cells = canvas_features(canvas, valid, float(s))
        window = torch.from_numpy(mf).to(DEVICE); shared = window_transforms(window, size)
        zn = zncc_everywhere(window, shared, size, torch.from_numpy(kernel).to(DEVICE), torch.from_numpy(cells.astype(np.float32)).to(DEVICE)).cpu().numpy()
        usable = (zn > -1) & inside; v = zn[usable]
        if usable.sum() < 100: continue
        flat = np.where(usable, zn, -9); best = np.unravel_index(np.argmax(flat), flat.shape)
        err = math.hypot(x_m[best[1]] - truth[0], y_m[best[0]] - truth[1]) / sigma_m
        z_best = (flat[best] - v.mean()) / v.std(); tn = np.where(usable & near_truth, zn, -9).max(); z_truth = (tn - v.mean()) / v.std() if tn > -9 else np.nan
        records.append(dict(photo=index, height=pose["height"], level=level, sigma_px=s, err=err, z_best=z_best, z_truth=z_truth))
    if index % 20 == 0: print(index, flush=True)
scale = pd.DataFrame(records); scale.to_csv(os.path.join(OUT, "scale_study.csv"), index=False)
scale["band"] = pd.cut(scale.height, [500, 800, 1100, 1400, 1700, 2100])
summary = scale.groupby(["band", "sigma_px"], observed=True).agg(n=("err", "size"), right2=("err", lambda e: (e <= 2).mean()), wrong=("err", lambda e: (e > 8).mean()),
          z_truth_med=("z_truth", "median")).round(2)
print(summary.unstack("sigma_px")[["right2", "wrong", "z_truth_med"]].to_string())
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col, title in zip(axes, ["right2", "wrong", "z_truth_med"], ["share with the best peak within 2 sigma", "share wrong (beyond 8 sigma)", "median z of the score near the TRUE place"]):
    table = summary[col].unstack("sigma_px")
    for band, rowv in table.iterrows(): ax.plot(table.columns, rowv.values, marker="o", label=str(band))
    ax.set_xlabel("gradient blur sigma (pixels of the map level)"); ax.set_title(title, fontsize=10); ax.grid(alpha=.3)
axes[0].legend(title="height (m)", fontsize=8); fig.tight_layout(); fig.savefig(os.path.join(OUT, "scale_study.png"), dpi=100)

20


40


60


140


160


200


... (3 outputs left out)

280


300


320


             right2                               wrong                               z_truth_med                               
sigma_px          1     2     3     4     6     8     1     2     3     4     6     8           1      2     3     4     6     8
band                                                                                                                            
(500, 800]     0.85  0.84  0.81  0.74  0.31  0.06  0.00  0.00  0.04  0.09  0.53  0.75       12.64  12.06  9.40  7.38  4.46  3.12
(800, 1100]    0.66  0.57  0.43  0.34  0.09  0.06  0.17  0.26  0.43  0.49  0.77  0.91        7.58   6.68  6.12  4.25  2.66  1.97
(1100, 1400]   0.80  0.45  0.38  0.20  0.10  0.00  0.00  0.40  0.55  0.78  0.88  1.00        7.96   5.70  3.87  2.95  2.26  0.83
(1400, 1700]   0.57  0.29  0.25  0.00  0.00  0.00  0.18  0.46  0.57  0.89  0.96  1.00        5.50   4.27  3.36  2.78  0.96  0.23
(1700, 2100]   0.38  0.36  0.26  0.05  0.05  0.03  0.44  0.49  0.69  0.90  0.95  0.97        3.11